In [ ]:
# ----------------------------------------------------------------------------
# 1. Imports
# ----------------------------------------------------------------------------
import os
import re
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

# ----------------------------------------------------------------------------
# 2. Paths
# ----------------------------------------------------------------------------
base_path = "/kaggle/input/emguka-trial-corpus/EMG-UKA-Trial-Corpus"
alignments_path = os.path.join(base_path, "Alignments")

# ----------------------------------------------------------------------------
# 3. Load Alignment (Phoneme) Data
# ----------------------------------------------------------------------------
def load_phoneme_sequences(data_path):
    data = {}
    for root, dirs, files in os.walk(data_path):
        for file in files:
            if file.endswith(".txt"):
                file_path = os.path.join(root, file)
                try:
                    with open(file_path, "r", encoding="utf-8") as f:
                        phonemes = f.read().strip().split()
                        utterance_id = os.path.splitext(file)[0]
                        data[utterance_id] = phonemes
                except Exception as e:
                    print(f"❌ Could not read {file_path}: {e}")
    return data

print("📂 Loading phoneme alignments...")
alignment_data = load_phoneme_sequences(alignments_path)
print(f"✅ Loaded {len(alignment_data)} phoneme sequences.")

# ----------------------------------------------------------------------------
# 4. Build Vocabulary
# ----------------------------------------------------------------------------
all_phonemes = []
for seq in alignment_data.values():
    all_phonemes.extend(seq)

unique_phonemes = sorted(list(set(all_phonemes)))
phoneme_to_idx = {p: i+1 for i, p in enumerate(unique_phonemes)}  # +1 for padding=0
idx_to_phoneme = {i: p for p, i in phoneme_to_idx.items()}
vocab_size = len(phoneme_to_idx) + 1  # include padding

print(f"📊 Vocabulary size: {vocab_size}")

# ----------------------------------------------------------------------------
# 5. Prepare Training Data (next phoneme prediction)
# ----------------------------------------------------------------------------
X, Y = [], []
for seq in alignment_data.values():
    encoded = [phoneme_to_idx[p] for p in seq if p in phoneme_to_idx]
    for i in range(1, len(encoded)):
        X.append(encoded[:i])
        Y.append(encoded[i])

max_len = max(len(x) for x in X)
X_padded = keras.preprocessing.sequence.pad_sequences(X, maxlen=max_len, padding="post")

Y = np.array(Y)

print(f"🟢 Training samples: {X_padded.shape[0]}, sequence length: {X_padded.shape[1]}")

# ----------------------------------------------------------------------------
# 6. Build Model
# ----------------------------------------------------------------------------
embedding_dim = 128
lstm_units = 256

model = keras.Sequential([
    layers.Embedding(input_dim=vocab_size, output_dim=embedding_dim, mask_zero=True),
    layers.Bidirectional(layers.LSTM(lstm_units, return_sequences=False)),
    layers.Dropout(0.3),
    layers.Dense(vocab_size, activation="softmax")
])

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

# ----------------------------------------------------------------------------
# 7. Train Model
# ----------------------------------------------------------------------------
history = model.fit(
    X_padded, Y,
    validation_split=0.1,
    batch_size=64,
    epochs=10,
    verbose=1
)

# ----------------------------------------------------------------------------
# 8. Plot Loss and Accuracy
# ----------------------------------------------------------------------------
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(history.history['loss'], label="Train Loss")
plt.plot(history.history['val_loss'], label="Val Loss")
plt.legend(); plt.title("Loss")

plt.subplot(1,2,2)
plt.plot(history.history['accuracy'], label="Train Acc")
plt.plot(history.history['val_accuracy'], label="Val Acc")
plt.legend(); plt.title("Accuracy")

plt.show()